# Dataset Construction: Methodology

This notebook documents the methodology for building the experimental dataset, detailing the rules, formulas, and judgement calls behind each standardised value.

The document outlines the general rules applicable to all sources. Decisions specific to individual sources are located in `notes/sources/[id].md`, and the complete source inventory is maintained in [data/source_log.csv](../data/source_log.csv).

The included code cells mirror the production modules found in the `src/` directory and execute small worked examples. This ensures the notebook remains self-contained and fully reproducible.


# 1. Pipeline Overview

Only three artifacts require modification per source; all other components are written once and reused.

<pre>
data/
  extracted/[id].csv    <span style="color:#b3b3b3">hand-transcribed data in the master schema (MANUAL)</span>
  processed/[id].csv    <span style="color:#b3b3b3">computed columns written by build.py (AUTOMATED)</span>
  excluded/             <span style="color:#b3b3b3">sources set aside from the model (e.g. angles)</span>
  master.csv            <span style="color:#b3b3b3">all processed sources concatenated (never edited by hand)</span>
  data_dictionary.md    <span style="color:#b3b3b3">column definitions (the schema contract)</span>
  source_log.csv        <span style="color:#b3b3b3">one summary row per source</span>

src/
  sections.py           <span style="color:#b3b3b3">section-type geometry (A I_major I_minor)</span>
  features.py           <span style="color:#b3b3b3">universal mechanics (r N_cr N_y lambda_bar chi)</span>
  classify.py           <span style="color:#b3b3b3">EN 1993-1-4 class + inferred failure mode</span>

build.py                <span style="color:#b3b3b3">orchestrates read → features → classify → write</span>

notes/sources/[id].md   <span style="color:#b3b3b3">per-source provenance + judgement calls</span>
</pre>


### Build Flow

The build fow is executed by `build.py` and remains identical for every run:

<pre>
extracted/[id].csv → add_features → classify → processed/[id].csv → concat → master.csv
</pre>




### Manual vs. Automated Processes

- **Manual (per source):** Reading the PDF, populating `notes/sources/[id].md`, typing rows into `extracted/[id].csv`  with converted units, and adding corresponding row to `source_log.csv`.
  
- **Automated (all sources concurrently):** The `build.py` script recomputes the `processed/` directory and `master.csv` from scratch. The script is invariant and functions purely based on the extracted CSV files, ensuring that re-running the process will not currupt the state.

The pipeline underwent two deliberate simplifications: an early YAML config-mapping layer and raw-CSV intermediaries were removed. Data is now typed directly into the master schema to minimize complexity unless a specific source necessitates it.

## 2. Source Selection and Structuring Rules

- **Experimental data only:** Many campaigns publish extensive FEA parametric sweeps. These act as algorithmically generated near-duplicates. Including these sweeps would artificially inflate model accuracy and cause severe data leakage. Consequently, only physical test rows are incorporated into the dataset.

- **Batch grouping:** Specimens sharing the same steel coil and coupon results are grouped by `study_id` to prevent leakage when combined with Grouped K-Fold or Leave-One-Study-Out methods rather than random splitting

- **Global flexural focus:** Local and yield-governed specimens are filtered out as described in Section 7. Interactive specimens (local and global) are labeled and retained for completeness but are flagged to ensure they are not fed into the pure-flexural model.

*Note on Angle and asymmetric sections*: These sections are excluded from the dataset. These typically fail via flexural-torsional buckling which is out of scope. The single angle source examined (Behzadi 2021) is stored in `data/excluded/`.

